<a href="https://colab.research.google.com/github/kinchittrivedi/Kaggle/blob/main/Modeling_with_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
jeffpoulshaju_feature_extraction_path = kagglehub.notebook_output_download('jeffpoulshaju/feature-extraction')

print('Data source import complete.')


Extracting files...
Data source import complete.


#  Modeling with XGBoost

Welcome to the third stage of our Intro to NLP Learn Guide! In our previous notebook, we successfully transformed our cleaned text and labels into a mathematical format. Our data now perfectly speaks the language of machine learning.

In this notebook, we will introduce **XGBoost**, a highly powerful machine learning algorithm. We will use our newly prepared data to **train this model** so it can learn the exact mathematical patterns that separate spam from normal messages. Once trained, we will immediately use it to **generate predictions.**

By the end of this notebook, we will have a fully trained model and its predictions saved and ready for the final performance evaluation. Let us load our data and get started!

In [3]:
import numpy as np
import scipy.sparse

# 1. Set your Kaggle input paths
X_path = f"{jeffpoulshaju_feature_extraction_path}/X_features.npz"
y_path = f"{jeffpoulshaju_feature_extraction_path}/y_encoded.npy"

# 2. Load the labels and features
y = np.load(y_path)
X = scipy.sparse.load_npz(X_path)

# Quick check to ensure the shapes match (Rows in X should equal items in y)
print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print("Data loaded successfully! Ready to train XGBoost.")

Features shape: (5169, 3000)
Labels shape: (5169,)
Data loaded successfully! Ready to train XGBoost.


## Splitting the Data

Before we teach our algorithm how to catch spam, we need to divide our dataset into two groups using a technique known as the **holdout method.** We will accomplish this using a standard **80 to 20 ratio.**

We will **feed 80% of our text messages directly into the algorithm.** This group is called the **training split** (which you will see in our upcoming code blocks as `X_train` and `y_train`). This large chunk of data acts as a textbook, allowing the model to study thousands of examples and learn exactly what makes a message normal or spam.

We intentionally **hold out the remaining 20% to serve as a rigorous final exam.** This group is known as the **test split** (represented as `X_test` and `y_test`). If we tested the model using the exact same data it just studied, it could easily get a perfect score by simply memorizing the answers. By forcing the model to evaluate completely unseen messages, we ensure it actually understands the underlying patterns and will perform reliably in the real world.

In [4]:
from sklearn.model_selection import train_test_split

# Split the data into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify the dimensions of our new datasets
print("--- Training Data Shape ---")
print(X_train.shape)
print("\n--- Testing Data Shape ---")
print(X_test.shape)

--- Training Data Shape ---
(4135, 3000)

--- Testing Data Shape ---
(1034, 3000)


## Modeling using XGBoost

We will now introduce XGBoost, which stands for **Extreme Gradient Boosting.** Instead of relying on a single massive equation, XGBoost uses a strategic approach called Boosting. It builds **a sequence of smaller models called decision trees.** The first tree analyzes the text data and inevitably makes mistakes. The algorithm then builds a second tree specifically to correct those errors, followed by a third to correct the second, and so on. By the end of training, **these sequential trees combine** into one highly accurate spam detection model.


To understand why this is so powerful, compare it to Bagging, or Bootstrap Aggregating. In a Bagging algorithm like a Random Forest, the **model builds hundreds of trees independently at the exact same time and simply averages their answers.** Because XGBoost learns sequentially from its past mistakes, it is often much more accurate for complex text classification.

![](https://github.com/jeffpoulshaju/Model-evaluation/raw/c6344021a330e8d59f8b366f72ed33f7d9697e19/Bagging%20vs%20Boosting.png)



## Class Imbalance

In our first notebook, we saw that our data is heavily unbalanced. We have 4,516 normal messages but only 653 spam messages. Because there is so much normal text, a basic model might just take the easy way out and guess that every single message is normal.

To fix this, we use a setting called **scale positive weight** and set it to 2. This simply gives the rare spam messages a louder voice, **telling the algorithm to treat every spam example as twice as important.** This ensures the model learns to catch spam without accidentally blocking your real emails. We also include a random state to guarantee our results are repeatable and set the evaluation metric to logloss to hide background warnings before we begin training.

Let's train the data using XGBoost using the code below.

In [5]:
# Install the XGBoost library
!pip install xgboost

from xgboost import XGBClassifier

# Initialize the model with parameters to equalize class imbalance, guarantee reproducible results, and suppress logging warnings
model = XGBClassifier(scale_pos_weight= 2,random_state=42, eval_metric='logloss')

# Train the model using our practice data
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

## Making Predictions

Now that our model has been trained on the patterns of spam, it is time to put it to the test. This is where we bring back the **20% of our data that we intentionally held out** earlier. We will pass these hidden testing features (`X_test`) into our model and ask it to predict whether each completely unseen message is **"ham" (0) or "spam" (1).** We will print the first 10 predictions to see the raw output.

In [6]:
# Use the trained model to predict labels for the test set
y_pred = model.predict(X_test)

# Display the first 10 predictions (0 = Ham, 1 = Spam)
print("--- First 10 Predictions ---")
print(y_pred[:10])

--- First 10 Predictions ---
[0 0 0 0 0 1 0 1 0 0]


## Saving Our Model and Predictions

Before we move on to the final evaluation stage of this Learn Guide, we need to save our work. In the next cell, we will use a library called Joblib to save our trained XGBoost model. This process freezes the complex logic and decision trees our model just learned so we can load it instantly in the future without needing to retrain it.

We will also save the actual test answers alongside the new predictions our model just generated. By saving these specific arrays right now, our final notebook will be incredibly clean and focused entirely on analyzing the performance metrics.

In [7]:
import numpy as np
import joblib

# 1. Save the trained XGBoost model
joblib.dump(model, 'xgboost_spam_model.pkl')

# 2. Save the actual answers and the model's predictions
np.save('y_test.npy', y_test)
np.save('y_pred.npy', y_pred)

print("Success! Model and predictions saved for the final evaluation.")

Success! Model and predictions saved for the final evaluation.


## What's Next?

Our XGBoost model is officially built and trained! Now, we need to see exactly how well it actually performs at catching spam.

Ready? Let's jump into the final step: [Evaluating NLP Models](https://www.kaggle.com/code/jeffpoulshaju/evaluating-nlp-models)